# Programación dinámica de dígitos

Para contar los números $0 \le x \le N$ que cumplen una propiedad de sus cifras, se construye $x$ cifra a cifra de izquierda a derecha. El estado guarda la posición $i$, un resumen de lo que importa de las cifras ya puestas (su suma, su resto módulo $m$, las cifras usadas…) y la marca **ajustado**: si hasta ahora $x$ coincide con el prefijo de $N$, la siguiente cifra no puede pasar de $N_i$; en cuanto se pone una cifra menor, el resto es libre y los subproblemas se repiten. Con $d$ cifras, $E$ resúmenes posibles y base 10, cuesta $O(d \cdot E \cdot 10)$ en lugar de $O(N)$.

Para un rango $[a, b]$ se calcula $f(b) - f(a - 1)$. Cada ejercicio se comprueba contra fuerza bruta para $N$ pequeños.

## 1. Números con suma de cifras $k$

Contar los $x \in [0, N]$ cuya suma de cifras es $k$ con una función memoizada `f(i, suma, ajustado)` sobre las cifras de $N$. Comprobarlo contra fuerza bruta para todos los $N \le 2000$ y algunos $k$, y calcular cuántos números de hasta 18 cifras suman 100.

In [1]:
import random
from functools import cache
from itertools import accumulate
from math import comb, perm


def suma_cifras(x):
    return sum(map(int, str(x)))


def cuenta_suma(N, k):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, suma, ajustado):
        if suma > k:
            return 0
        if i == len(cifras):
            return int(suma == k)
        tope = cifras[i] if ajustado else 9
        return sum(f(i + 1, suma + c, ajustado and c == tope) for c in range(tope + 1))

    return f(0, 0, True)


for N in range(2001):
    for k in (0, 1, 5, 10, 19):
        assert cuenta_suma(N, k) == sum(suma_cifras(x) == k for x in range(N + 1))

print("Números de 0 a 999 que suman 13:", cuenta_suma(999, 13))
print("Números menores que 10^18 cuyas cifras suman 100:", cuenta_suma(10**18 - 1, 100))
print("Números en [2026, 2026000] que suman 26:", cuenta_suma(2_026_000, 26) - cuenta_suma(2025, 26))
assert cuenta_suma(999, 13) == 75

Números de 0 a 999 que suman 13: 75
Números menores que 10^18 cuyas cifras suman 100: 9869362547411187
Números en [2026, 2026000] que suman 26: 109143


## 2. El $k$-ésimo número con suma de cifras $s$

Usar el conteo para encontrar, sin recorrerlos todos, el $k$-ésimo número (empezando en 1) cuya suma de cifras es $s$: primero buscar el número de cifras necesario y después fijar las cifras de izquierda a derecha, eligiendo la menor cifra $c$ tal que los números que empiezan por el prefijo con $c$ ya acumulan $k$. Comprobarlo con una lista generada por fuerza bruta.

In [2]:
@cache
def completar(longitud, suma):
    """Formas de rellenar `longitud` cifras libres (0-9) que sumen `suma`."""
    if suma < 0:
        return 0
    if longitud == 0:
        return int(suma == 0)
    return sum(completar(longitud - 1, suma - c) for c in range(10))


def k_esimo_con_suma(k, s):
    longitud = 1
    while completar(longitud, s) < k:  # números con a lo sumo `longitud` cifras
        longitud += 1
    resultado, resto = 0, s
    for i in range(longitud):
        for c in range(10):
            formas = completar(longitud - i - 1, resto - c)
            if formas >= k:
                resultado, resto = resultado * 10 + c, resto - c
                break
            k -= formas
    return resultado


for s in (1, 7, 15):
    lista = [x for x in range(1, 200_000) if suma_cifras(x) == s]
    for k in range(1, min(len(lista), 500) + 1):
        assert k_esimo_con_suma(k, s) == lista[k - 1]
print("Los 10 primeros que suman 7:", [k_esimo_con_suma(k, 7) for k in range(1, 11)])
print("El número 1 000 000 que suma 50:", k_esimo_con_suma(10**6, 50))
assert suma_cifras(k_esimo_con_suma(10**6, 50)) == 50
assert cuenta_suma(k_esimo_con_suma(10**6, 50), 50) == 10**6

Los 10 primeros que suman 7:

 [7, 16, 25, 34, 43, 52, 61, 70, 106, 115]
El número 1 000 000 que suma 50: 95795744


## 3. Números sin el dígito 4

En algunos edificios no hay planta 4 ni ninguna que lo contenga (14, 24, 40…). Contar los números de $1$ a $N$ sin ningún 4 de dos formas: con programación dinámica de dígitos y con una fórmula (cada cifra de $N$ aporta $c_i \cdot 9^{\text{cifras restantes}}$ si se baja de $c_i$, sin contar el 4, y se corta al llegar a un 4 de $N$). Después, calcular qué número aparece en el botón de la planta real $k$ del ascensor: es $k$ escrito en base 9 cambiando las cifras $4, \dots, 8$ por $5, \dots, 9$.

In [3]:
def sin_cuatro_pd(N):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, ajustado):
        if i == len(cifras):
            return 1
        tope = cifras[i] if ajustado else 9
        return sum(f(i + 1, ajustado and c == tope) for c in range(tope + 1) if c != 4)

    return f(0, True) - 1  # quitar el 0


def sin_cuatro_formula(N):
    cifras = list(map(int, str(N)))
    total = 0
    for i, c in enumerate(cifras):
        menores = c - (c > 4)
        total += menores * 9 ** (len(cifras) - i - 1)
        if c == 4:
            return total - 1  # se quita el 0
    return total  # +1 por el propio N, -1 por el 0


def boton(planta):
    cifras = []
    while planta:
        planta, r = divmod(planta, 9)
        cifras.append(str(r + (r >= 4)))
    return int("".join(reversed(cifras)))


botones = [x for x in range(1, 10_000) if "4" not in str(x)]
for N in range(1, 10_000):
    assert sin_cuatro_pd(N) == sin_cuatro_formula(N) == sum(1 for x in botones if x <= N)
for planta in range(1, len(botones) + 1):
    assert boton(planta) == botones[planta - 1]

print("Plantas sin 4 hasta el 100:", sin_cuatro_pd(100))
print("Botones de las plantas reales 1 a 15:", [boton(k) for k in range(1, 16)])
print("Números sin 4 hasta 10^15:", sin_cuatro_formula(10**15))
assert sin_cuatro_formula(10**15 - 1) == 9**15 - 1

Plantas sin 4 hasta el 100: 81
Botones de las plantas reales 1 a 15: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17]
Números sin 4 hasta 10^15: 205891132094649


## 4. Números sin cifras repetidas

Contar los números de $1$ a $N$ cuyas cifras son todas distintas. El estado guarda la máscara de cifras usadas, pero los ceros a la izquierda no cuentan como cifras, así que hace falta otra marca `empezado`. Comprobarlo contra fuerza bruta y contra la fórmula combinatoria para $N = 10^d - 1$: $\sum_{l=1}^{\min(d, 10)} 9 \cdot \frac{9!}{(10-l)!}$ (9 opciones para la primera cifra y variaciones de las 9 restantes).

In [4]:
def sin_repetidas(N):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, mascara, ajustado, empezado):
        if i == len(cifras):
            return int(empezado)
        tope = cifras[i] if ajustado else 9
        total = 0
        for c in range(tope + 1):
            sigue_ajustado = ajustado and c == tope
            if not empezado and c == 0:
                total += f(i + 1, mascara, sigue_ajustado, False)
            elif not mascara >> c & 1:
                total += f(i + 1, mascara | 1 << c, sigue_ajustado, True)
        return total

    return f(0, 0, True, False)


def todas_distintas(x):
    return len(set(str(x))) == len(str(x))


for N in random.Random(1).sample(range(1, 100_000), 300) + list(range(1, 300)):
    assert sin_repetidas(N) == sum(todas_distintas(x) for x in range(1, N + 1))
for d in range(1, 12):
    formula = sum(9 * perm(9, l - 1) for l in range(1, min(d, 10) + 1))
    assert sin_repetidas(10**d - 1) == formula

print(f"{'hasta':>14} {'sin repetidas':>14}")
for d in range(1, 12):
    print(f"{10**d - 1:>14} {sin_repetidas(10**d - 1):>14}")

         hasta  sin repetidas
             9              9
            99             90
           999            738
          9999           5274
         99999          32490
        999999         168570
       9999999         712890
      99999999        2345850
     999999999        5611770
    9999999999        8877690
   99999999999        8877690


## 5. Múltiplos de $m$ y números de Harshad

Contar los $x \in [1, N]$ múltiplos de $m$ guardando en el estado el resto módulo $m$ del prefijo ($r \to (10r + c) \bmod m$) y comprobar que sale $\lfloor N/m \rfloor$. Después, contar los números de Harshad (divisibles por la suma de sus cifras, como 2025, cuyas cifras suman 9, y $2025 = 9 \cdot 225$): para cada suma posible $s$ se cuenta con estado (suma, resto módulo $s$) y se suman los resultados.

In [5]:
def multiplos(N, m):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, resto, ajustado):
        if i == len(cifras):
            return int(resto == 0)
        tope = cifras[i] if ajustado else 9
        return sum(f(i + 1, (10 * resto + c) % m, ajustado and c == tope) for c in range(tope + 1))

    return f(0, 0, True) - 1  # el 0 es múltiplo de todo


def harshad(N):
    cifras = list(map(int, str(N)))
    total = 0
    for s in range(1, 9 * len(cifras) + 1):

        @cache
        def f(i, suma, resto, ajustado):
            if suma > s:
                return 0
            if i == len(cifras):
                return int(suma == s and resto == 0)
            tope = cifras[i] if ajustado else 9
            return sum(f(i + 1, suma + c, (10 * resto + c) % s, ajustado and c == tope) for c in range(tope + 1))

        total += f(0, 0, 0, True)
    return total


rng = random.Random(2)
for _ in range(300):
    N, m = rng.randint(1, 10**9), rng.randint(1, 60)
    assert multiplos(N, m) == N // m

es_harshad = [x % suma_cifras(x) == 0 for x in range(1, 1_000_001)]
cuentas = list(accumulate(es_harshad))
for N in list(range(1, 200)) + rng.sample(range(1, 20_001), 40):
    assert harshad(N) == cuentas[N - 1]
print("2025 es de Harshad:", 2025 % suma_cifras(2025) == 0, " 2026:", 2026 % suma_cifras(2026) == 0)
print("Números de Harshad hasta 10^4:", harshad(10**4))
assert harshad(10**6) == cuentas[-1]
print("Números de Harshad hasta 10^9:", harshad(10**9))

2025 es de Harshad: True  2026: False
Números de Harshad hasta 10^4: 1538


Números de Harshad hasta 10^9: 61574510


## 6. Suma de todas las cifras de $1$ a $N$

Calcular $\sum_{x=1}^{N} \text{suma\_cifras}(x)$ sin recorrer los números: cada estado devuelve un par (cuántos números completan el prefijo, suma de las cifras que aportan), y al poner la cifra $c$ se suma $c$ por cada número que la contiene. Comprobarlo contra fuerza bruta y contra la fórmula $45 \cdot d \cdot 10^{d-1}$ para $N = 10^d - 1$.

In [6]:
def suma_de_cifras_hasta(N):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, ajustado):
        if i == len(cifras):
            return 1, 0
        tope = cifras[i] if ajustado else 9
        cuantos = suma = 0
        for c in range(tope + 1):
            n, s = f(i + 1, ajustado and c == tope)
            cuantos += n
            suma += s + c * n
        return cuantos, suma

    return f(0, True)[1]


acumuladas = list(accumulate(suma_cifras(x) for x in range(100_001)))
for N in list(range(1, 2000)) + random.Random(3).sample(range(1, 100_001), 200):
    assert suma_de_cifras_hasta(N) == acumuladas[N]
for d in range(1, 19):
    assert suma_de_cifras_hasta(10**d - 1) == 45 * d * 10 ** (d - 1)
print("Suma de las cifras de 1 a 2026:", suma_de_cifras_hasta(2026))
print("Suma de las cifras de 1 a 10^18:", suma_de_cifras_hasta(10**18))

Suma de las cifras de 1 a 2026: 28189
Suma de las cifras de 1 a 10^18: 81000000000000000001


## 7. Cuántas veces aparece cada cifra

Para numerar las 1500 páginas de una guía de senderos de La Gomera, la imprenta necesita saber cuántos tipos de cada cifra usa. Contar las apariciones de cada cifra $d$ en todos los números de $1$ a $N$; para el 0 no cuentan los ceros a la izquierda, lo que vuelve a pedir la marca `empezado`. Comprobar con un `Counter` sobre los números escritos y que el total coincide con el número de cifras escritas.

In [7]:
from collections import Counter


def apariciones(N, d):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, ajustado, empezado):
        if i == len(cifras):
            return 1, 0
        tope = cifras[i] if ajustado else 9
        cuantos = veces = 0
        for c in range(tope + 1):
            ahora = empezado or c != 0
            n, v = f(i + 1, ajustado and c == tope, ahora)
            cuantos += n
            veces += v + n * (ahora and c == d)
        return cuantos, veces

    return f(0, True, False)[1]


def apariciones_bruto(N):
    return Counter("".join(map(str, range(1, N + 1))))


cuenta = apariciones_bruto(1500)
print("Cifras para numerar 1500 páginas:")
for d in range(10):
    print(f"  {d}: {apariciones(1500, d):>4}")
total = sum(apariciones(1500, d) for d in range(10))
print("Total de cifras:", total)
assert all(apariciones(1500, d) == cuenta[str(d)] for d in range(10))
assert total == sum(len(str(x)) for x in range(1, 1501))
for N in random.Random(4).sample(range(1, 30_000), 60):
    cuenta = apariciones_bruto(N)
    assert all(apariciones(N, d) == cuenta[str(d)] for d in range(10))

Cifras para numerar 1500 páginas:
  0:  391
  1: 1001
  2:  500
  3:  500
  4:  500
  5:  401
  6:  400
  7:  400
  8:  400
  9:  400
Total de cifras: 4893


## 8. Cifras en orden no decreciente

Contar los números de $1$ a $N$ cuyas cifras no bajan nunca de izquierda a derecha (como 1123 o 4799). El estado guarda la última cifra puesta, y basta con que los ceros a la izquierda no cuenten (las cifras de un número positivo no decreciente nunca son 0 salvo el propio cero). Comprobarlo contra fuerza bruta y contra la fórmula de combinaciones con repetición para $N = 10^d - 1$: hay $\binom{d + 9}{9} - 1$, porque cada número es un multiconjunto de $d$ cifras de 0 a 9 completado con ceros delante.

In [8]:
def no_decrecientes(N):
    cifras = list(map(int, str(N)))

    @cache
    def f(i, ultima, ajustado):
        if i == len(cifras):
            return 1
        tope = cifras[i] if ajustado else 9
        return sum(f(i + 1, c, ajustado and c == tope) for c in range(ultima, tope + 1))

    return f(0, 0, True) - 1


def es_no_decreciente(x):
    s = str(x)
    return all(a <= b for a, b in zip(s, s[1:]))


for N in list(range(1, 3000)) + random.Random(5).sample(range(1, 300_000), 50):
    assert no_decrecientes(N) == sum(es_no_decreciente(x) for x in range(1, N + 1))
for d in range(1, 19):
    assert no_decrecientes(10**d - 1) == comb(d + 9, 9) - 1
print("No decrecientes hasta 2026:", no_decrecientes(2026))
print("No decrecientes con hasta 18 cifras:", no_decrecientes(10**18 - 1))

No decrecientes hasta 2026: 384
No decrecientes con hasta 18 cifras: 4686824


## 9. Sin dos cifras vecinas iguales en un rango

Un número de abonado del bono de guagua es «cómodo de dictar» si no tiene dos cifras consecutivas iguales (como 12121, y no 1223). Contar los números cómodos en el rango $[a, b]$ con $f(b) - f(a - 1)$, guardando la cifra anterior (o «ninguna» mientras no haya empezado el número). Comprobarlo contra fuerza bruta en rangos aleatorios y contra $9^d$ para los números de exactamente $d$ cifras.

In [9]:
def comodos(N):
    if N < 0:
        return 0
    cifras = list(map(int, str(N)))

    @cache
    def f(i, anterior, ajustado):
        if i == len(cifras):
            return 1
        tope = cifras[i] if ajustado else 9
        total = 0
        for c in range(tope + 1):
            if anterior == -1 and c == 0:
                total += f(i + 1, -1, ajustado and c == tope)
            elif c != anterior:
                total += f(i + 1, c, ajustado and c == tope)
        return total

    return f(0, -1, True)  # incluye el 0


def comodos_rango(a, b):
    return comodos(b) - comodos(a - 1)


def es_comodo(x):
    s = str(x)
    return all(p != q for p, q in zip(s, s[1:]))


rng = random.Random(6)
for _ in range(200):
    a = rng.randint(0, 50_000)
    b = rng.randint(a, a + 5000)
    assert comodos_rango(a, b) == sum(es_comodo(x) for x in range(a, b + 1))
for d in range(1, 16):
    assert comodos_rango(10 ** (d - 1), 10**d - 1) == 9**d
print("Cómodos entre 100000 y 999999:", comodos_rango(100_000, 999_999))
print("Cómodos entre 2026 y 20262026:", comodos_rango(2026, 20_262_026))

Cómodos entre 100000 y 999999: 531441
Cómodos entre 2026 y 20262026: 10255573


## 10. Una plantilla genérica

Todos los ejercicios repiten el mismo esqueleto. Escribir `cuenta_digitos(N, inicial, transicion, acepta)`, donde `transicion(estado, cifra, empezado, posicion)` devuelve el nuevo estado (o `None` si la cifra no se permite) y `acepta(estado)` dice si un número completo vale. Rehacer con ella los ejercicios de suma $k$, sin el 4, sin repetidas y múltiplos de $m$, y comprobar que dan lo mismo que las versiones específicas. Usarla también para un caso nuevo: números cuya suma de cifras en posiciones pares es igual a la de posiciones impares (contando desde la derecha, son los múltiplos de 11 con diferencia 0, como 121 o 2002).

In [10]:
def cuenta_digitos(N, inicial, transicion, acepta):
    cifras = list(map(int, str(N)))
    L = len(cifras)

    @cache
    def f(i, estado, ajustado, empezado):
        if i == L:
            return int(empezado and acepta(estado))
        tope = cifras[i] if ajustado else 9
        total = 0
        for c in range(tope + 1):
            ahora = empezado or c != 0
            nuevo = transicion(estado, c, ahora, L - 1 - i) if ahora else estado
            if nuevo is not None:
                total += f(i + 1, nuevo, ajustado and c == tope, ahora)
        return total

    return f(0, inicial, True, False)


def generica_suma(N, k):
    return cuenta_digitos(N, 0, lambda s, c, e, p: s + c if s + c <= k else None, lambda s: s == k)


def generica_sin_cuatro(N):
    return cuenta_digitos(N, 0, lambda s, c, e, p: None if c == 4 else s, lambda s: True)


def generica_sin_repetidas(N):
    return cuenta_digitos(N, 0, lambda s, c, e, p: None if s >> c & 1 else s | 1 << c, lambda s: True)


def generica_multiplos(N, m):
    return cuenta_digitos(N, 0, lambda r, c, e, p: (10 * r + c) % m, lambda r: r == 0)


def equilibrados(N):
    # p es la posición desde la derecha: las pares suman y las impares restan
    return cuenta_digitos(N, 0, lambda d, c, e, p: d + c if p % 2 == 0 else d - c, lambda d: d == 0)


rng = random.Random(7)
for _ in range(100):
    N = rng.randint(1, 10**6)
    k, m = rng.randint(1, 30), rng.randint(1, 50)
    assert generica_suma(N, k) == cuenta_suma(N, k) - (k == 0)
    assert generica_sin_cuatro(N) == sin_cuatro_pd(N)
    assert generica_sin_repetidas(N) == sin_repetidas(N)
    assert generica_multiplos(N, m) == multiplos(N, m)


def es_equilibrado(x):
    s = str(x)[::-1]
    return sum(map(int, s[0::2])) == sum(map(int, s[1::2]))


for N in rng.sample(range(1, 50_000), 60):
    assert equilibrados(N) == sum(es_equilibrado(x) for x in range(1, N + 1))
print("Equilibrados hasta 10^4:", equilibrados(10**4), [x for x in range(1, 2100) if es_equilibrado(x)][:8], "…")
print("Equilibrados hasta 10^12:", equilibrados(10**12))
assert all(x % 11 == 0 for x in range(1, 50_000) if es_equilibrado(x))

Equilibrados hasta 10^4: 669 [11, 22, 33, 44, 55, 66, 77, 88] …
Equilibrados hasta 10^12: 39581170419
